Here we get the $F_{ij}$ fidelities from simulation - the 'fake providers' like FakeToronto basically use a snapshot of a system, see [the documentation](https://quantum.cloud.ibm.com/docs/en/api/qiskit-ibm-runtime/fake-provider). First i get them and look at the standard deviation to find a good one to run our benchmarks on. the larger the standard deviation the better it should work. im specifically looking for cx hardware because i know that cx is two applications of iswap

In [1]:
from qiskit_ibm_runtime.fake_provider import FakeProviderForBackendV2
import numpy as np

provider = FakeProviderForBackendV2()

print(f"{'Backend':<30} {'Qubits':>7} {'Min F':>7} {'Max F':>7} "
      f"{'Std F':>7} {'Range':>7}")
print("-" * 70)

for backend in sorted(provider.backends(), key=lambda x: x.name):
    try:
        # Get the 2Q gate
        two_q = [op for op in backend.operation_names
                 if op in ('cx', 'ecr', 'cz')]
        if not two_q:
            continue
        gate = two_q[0]

        # Extract error rates
        errors = []
        for qargs, props in backend.target[gate].items():
            if (qargs is not None and len(qargs) == 2
                    and props is not None
                    and props.error is not None
                    and 0 < props.error < 0.5):
                errors.append(props.error)

        if len(errors) < 10:
            continue

        fids = [np.sqrt(1 - e) if gate != 'ecr'
                else np.sqrt(1 - e) for e in errors]
        std = np.std(fids)
        print(f"{backend.name:<30} {backend.num_qubits:>7} "
              f"{min(fids):>7.4f} {max(fids):>7.4f} "
              f"{std:>7.4f} {max(fids)-min(fids):>7.4f}")
    except Exception:
        continue

/home/dylan/qiskit/lib/python3.10/site-packages/qiskit_ibm_runtime/fake_provider/backends/nighthawk/fake_nighthawk.py:76: UserWarning: Properties of fake_nighthawk are not intended to represent typical nighthawk error values.
  warnings.warn(


Backend                         Qubits   Min F   Max F   Std F   Range
----------------------------------------------------------------------
fake_algiers                        27  0.9728  0.9977  0.0060  0.0250
fake_almaden                        20  0.9619  0.9945  0.0085  0.0326
fake_auckland                       27  0.9883  0.9974  0.0021  0.0092
fake_boeblingen                     20  0.9860  0.9969  0.0028  0.0109
fake_brisbane                      127  0.9561  0.9982  0.0070  0.0421
fake_brooklyn                       65  0.9738  0.9968  0.0043  0.0230
fake_cairo                          27  0.9896  0.9962  0.0020  0.0066
fake_cambridge                      28  0.9562  0.9930  0.0094  0.0368
fake_casablanca                      7  0.9908  0.9962  0.0019  0.0055
fake_cusco                         127  0.8951  0.9972  0.0184  0.1022
fake_fez                           156  0.9747  0.9989  0.0030  0.0243
fake_geneva                         27  0.9418  0.9982  0.0083  0.0564
fake_g

once we have the fidelities for $CX$, we transform them into fidelities for $\sqrt{iSWAP}$ 

In [3]:
from qiskit_ibm_runtime.fake_provider import FakeTorontoV2
import numpy as np

backend = FakeTorontoV2()
target = backend.target

# Extract CX errors
cx_errors = {}
for qargs, props in target["cx"].items():
    if (qargs is not None and len(qargs) == 2
            and props is not None
            and props.error is not None
            and 0 < props.error < 0.5):
        cx_errors[qargs] = props.error

print(f"Backend: {backend.name}, {backend.num_qubits} qubits")
print(f"CX links: {len(cx_errors)}")
print(f"\nCX error → sqrt(iSWAP) fidelity conversion:")
print(f"{'Link':<12} {'CX error':>10} {'F_cx':>8} {'F_sqiswap':>10}")
print("-" * 45)

for (p0, p1), e in sorted(cx_errors.items())[:10]:
    f_cx = 1 - e
    f_sqiswap = np.sqrt(f_cx)
    print(f"({p0:>2},{p1:>2})      {e:>10.4f} {f_cx:>8.4f} {f_sqiswap:>10.4f}")

errors = list(cx_errors.values())
f_sqiswap_all = [np.sqrt(1 - e) for e in errors]
print(f"\nSqrt(iSWAP) fidelity stats:")
print(f"  min:  {min(f_sqiswap_all):.4f}  (worst link CX error: "
      f"{max(errors):.4f})")
print(f"  max:  {max(f_sqiswap_all):.4f}  (best link CX error:  "
      f"{min(errors):.4f})")
print(f"  mean: {np.mean(f_sqiswap_all):.4f}")
print(f"  std:  {np.std(f_sqiswap_all):.4f}")
print(f"  range:{max(f_sqiswap_all)-min(f_sqiswap_all):.4f}")

# Sanity check: verify sqrt relationship
print(f"\nSanity check:")
print(f"  If F_sqiswap = {min(f_sqiswap_all):.4f}, then F_cx = "
      f"{min(f_sqiswap_all)**2:.4f} "
      f"(CX error = {1 - min(f_sqiswap_all)**2:.4f})")
print(f"  Actual worst CX error: {max(errors):.4f}" 
      if abs(1 - min(f_sqiswap_all)**2 - max(errors)) < 1e-6
      else f"  Actual worst CX error: {max(errors):.4f} — mismatch!")

Backend: fake_toronto, 27 qubits
CX links: 56

CX error → sqrt(iSWAP) fidelity conversion:
Link           CX error     F_cx  F_sqiswap
---------------------------------------------
( 0, 1)          0.0089   0.9911     0.9955
( 1, 0)          0.0089   0.9911     0.9955
( 1, 2)          0.0127   0.9873     0.9937
( 1, 4)          0.0073   0.9927     0.9964
( 2, 1)          0.0127   0.9873     0.9937
( 2, 3)          0.0078   0.9922     0.9961
( 3, 2)          0.0078   0.9922     0.9961
( 3, 5)          0.0092   0.9908     0.9954
( 4, 1)          0.0073   0.9927     0.9964
( 4, 7)          0.0087   0.9913     0.9956

Sqrt(iSWAP) fidelity stats:
  min:  0.9338  (worst link CX error: 0.1280)
  max:  0.9971  (best link CX error:  0.0057)
  mean: 0.9895
  std:  0.0125
  range:0.0633

Sanity check:
  If F_sqiswap = 0.9338, then F_cx = 0.8720 (CX error = 0.1280)
  Actual worst CX error: 0.1280


Now the full protocol - note here we find the best layout with out fidelity matrix. i should probably put this into a step in the transpilation but i havent yet

In [4]:
from qiskit_ibm_runtime.fake_provider import FakeTorontoV2
import numpy as np
import urllib.request
import tempfile
import os
import copy
from qiskit import qasm2
from qiskit.transpiler import PassManager, CouplingMap, Layout
from qiskit.transpiler.passes import (
    SetLayout, FullAncillaAllocation, EnlargeWithAncilla, ApplyLayout,
    Optimize1qGatesDecomposition, HighLevelSynthesis,
    UnrollCustomDefinitions, BasisTranslator,
    Collect2qBlocks, ConsolidateBlocks,
    SabreSwap,
)
from qiskit.circuit.equivalence_library import SessionEquivalenceLibrary
from qiskit.converters import circuit_to_dag
from qiskit.dagcircuit import DAGOpNode
from qiskit.quantum_info import Operator
from mqt.bench import BenchmarkLevel, get_benchmark
from mirage import MirageSwap, MirageDecompose
from mirage.cost import decomp_cost

# ---------------------------------------------------------------------------
# Build F matrix and coupling map from FakeToronto calibration data
# ---------------------------------------------------------------------------

backend = FakeTorontoV2()
target = backend.target

cx_errors = {}
for qargs, props in target["cx"].items():
    if (qargs is not None and len(qargs) == 2
            and props is not None
            and props.error is not None
            and 0 < props.error < 0.5):
        cx_errors[qargs] = props.error

calibrated_neighbors = {}
for (p0, p1) in cx_errors:
    calibrated_neighbors.setdefault(p0, []).append(p1)
    calibrated_neighbors.setdefault(p1, []).append(p0)

best_root = max(calibrated_neighbors,
                key=lambda q: len(calibrated_neighbors[q]))
visited = []
queue = [best_root]
seen = {best_root}
while queue:
    q = queue.pop(0)
    visited.append(q)
    for nb in calibrated_neighbors.get(q, []):
        if nb not in seen:
            seen.add(nb)
            queue.append(nb)

selected = set(visited)
phys_to_idx = {phys: idx for idx, phys in enumerate(visited)}
n = len(visited)

subgraph_edges = [
    (phys_to_idx[p0], phys_to_idx[p1])
    for (p0, p1) in cx_errors
    if p0 in selected and p1 in selected
]
coupling_map = CouplingMap(couplinglist=subgraph_edges)

F = np.eye(n)
for (p0, p1), e in cx_errors.items():
    if p0 in phys_to_idx and p1 in phys_to_idx:
        i = phys_to_idx[p0]
        j = phys_to_idx[p1]
        f = np.sqrt(1.0 - e)
        F[i, j] = f
        F[j, i] = f

fids_per_qubit = []
for i in range(n):
    nbs = list(coupling_map.neighbors(i))
    if nbs:
        fids_per_qubit.append((i, np.mean([F[i, nb] for nb in nbs]), visited[i]))
fids_per_qubit.sort(key=lambda x: x[1])

fids_on_links = [F[i, j] for i, j in coupling_map.get_edges() if F[i, j] > 0]
print(f"FakeToronto {n}-qubit heavy-hex")
print(f"F_sqiswap: min={min(fids_on_links):.4f}  max={max(fids_on_links):.4f}"
      f"  mean={np.mean(fids_on_links):.4f}  std={np.std(fids_on_links):.4f}")
print("\nWorst 5 qubits:")
for idx, avg, phys in fids_per_qubit[:5]:
    print(f"  logical {idx:>2} (physical {phys:>2}): avg_F={avg:.4f}")
print("Best 5 qubits:")
for idx, avg, phys in fids_per_qubit[-5:]:
    print(f"  logical {idx:>2} (physical {phys:>2}): avg_F={avg:.4f}")


# ---------------------------------------------------------------------------
# Helper functions
# ---------------------------------------------------------------------------

def make_unroll_consolidate():
    return [
        HighLevelSynthesis(),
        UnrollCustomDefinitions(SessionEquivalenceLibrary,
                                basis_gates=["u", "cx", "swap"]),
        BasisTranslator(SessionEquivalenceLibrary,
                        target_basis=["u", "cx", "swap"]),
        Collect2qBlocks(),
        ConsolidateBlocks(),
    ]


def circuit_depth_sqrt_iswap(circuit):
    dag = circuit_to_dag(circuit)
    depth_at = {}
    for node in dag.topological_op_nodes():
        cost = 1.0 if node.op.name == "xx_plus_yy" else 0.0
        max_pred = 0.0
        for pred in dag.quantum_predecessors(node):
            if isinstance(pred, DAGOpNode):
                v = depth_at.get(pred._node_id, 0.0)
                if v > max_pred:
                    max_pred = v
        depth_at[node._node_id] = max_pred + cost
    return max(depth_at.values()) if depth_at else 0.0


def fidelity_cost(circuit, F):
    dag = circuit_to_dag(circuit)
    total = 0.0
    for node in dag.topological_op_nodes():
        if len(node.qargs) != 2:
            continue
        p0 = dag.find_bit(node.qargs[0]).index
        p1 = dag.find_bit(node.qargs[1]).index
        if node.op.name == "xx_plus_yy":
            k = 1.0
        elif node.op.name == "swap":
            k = 3.0
        elif node.op.name == "unitary":
            try:
                k = float(decomp_cost(Operator(node.op).data))
            except Exception:
                k = 2.0
        else:
            k = 0.0
        f = max(float(F[p0, p1]), 1e-10)
        total += k * (-np.log(f))
    return total


def best_fidelity_layout(circuit, coupling_map, F):
    n_virtual = circuit.num_qubits
    n_physical = coupling_map.size()
    scores = np.zeros(n_physical)
    for p in range(n_physical):
        nbs = list(coupling_map.neighbors(p))
        if nbs:
            scores[p] = np.mean([F[p, nb] for nb in nbs])
    best_physical = np.argsort(scores)[::-1][:n_virtual]
    layout = Layout()
    for virt_idx, phys_idx in enumerate(best_physical):
        layout[circuit.qubits[virt_idx]] = int(phys_idx)
    return layout


BASE = "https://raw.githubusercontent.com/pnnl/QASMBench/master/"


def load_qasmbench(name, size):
    url = f"{BASE}{size}/{name}/{name}.qasm"
    with urllib.request.urlopen(url) as r:
        qasm_str = r.read().decode("utf-8")
    with tempfile.NamedTemporaryFile(
            suffix=".qasm", mode="w", delete=False) as f:
        f.write(qasm_str)
        tmp = f.name
    qc = qasm2.load(tmp)
    os.unlink(tmp)
    return qc


def run_both(qc, label):
    F_tuple = tuple(map(tuple, F))
    layout = best_fidelity_layout(qc, coupling_map, F)

    results = {}
    for mode, consider_fid in [("depth", False), ("fidelity", True)]:
        mirage = MirageSwap(
            coupling_map,
            n_trials=20,
            seed=42,
            consider_fidelities=consider_fid,
            fidelity_matrix=F_tuple if consider_fid else None,
        )
        pm = PassManager(
            make_unroll_consolidate() + [
                SetLayout(layout),
                FullAncillaAllocation(coupling_map),
                EnlargeWithAncilla(),
                ApplyLayout(),
                mirage,
                MirageDecompose(),
                Optimize1qGatesDecomposition(basis=["rz", "ry", "rx"]),
            ]
        )
        routed = pm.run(qc)
        results[mode] = {
            "depth": circuit_depth_sqrt_iswap(routed),
            "cost":  fidelity_cost(routed, F),
            "gates": routed.count_ops().get("xx_plus_yy", 0),
        }

    d = results["depth"]
    f = results["fidelity"]
    fid_impr  = (d["cost"] - f["cost"]) / d["cost"] * 100
    depth_pen = (f["depth"] - d["depth"]) / d["depth"] * 100 \
                if d["depth"] > 0 else 0

    print(f"  depth-sel:    depth={d['depth']:.0f}  "
          f"cost={d['cost']:.4f}  gates={d['gates']}")
    print(f"  fidelity-sel: depth={f['depth']:.0f}  "
          f"cost={f['cost']:.4f}  gates={f['gates']}")
    print(f"  → fid_impr={fid_impr:.1f}%  depth_pen={depth_pen:.1f}%",
          flush=True)

    return d["depth"], f["depth"], d["cost"], f["cost"], fid_impr, depth_pen


# ---------------------------------------------------------------------------
# Benchmark
# ---------------------------------------------------------------------------

CIRCUITS = [
    ("qft_n18",          "mqt",       ("qft",            18)),
    ("qftentangled_n16", "mqt",       ("qftentangled",   16)),
    ("qpeexact_n16",     "mqt",       ("qpeexact",       16)),
    ("ae_n16",           "mqt",       ("ae",             16)),
    ("multiplier_n15",   "qasmbench", ("multiplier_n15", "medium")),
    ("bigadder_n18",     "qasmbench", ("bigadder_n18",   "medium")),
    ("qec9xz_n17",       "qasmbench", ("qec9xz_n17",     "medium")),
    ("seca_n11",         "qasmbench", ("seca_n11",       "medium")),
    ("sat_n11",          "qasmbench", ("sat_n11",        "medium")),
    ("wstate_n27",       "qasmbench", ("wstate_n27",     "medium")),
]

print("\n" + "=" * 75)
all_results = []
for name, source, args in CIRCUITS:
    print(f"\n--- {name} ---", flush=True)
    try:
        if source == "mqt":
            qc = get_benchmark(args[0], BenchmarkLevel.ALG, args[1])
        else:
            qc = load_qasmbench(args[0], args[1])
        if qc.num_qubits > coupling_map.size():
            print(f"  SKIP: {qc.num_qubits} qubits > {coupling_map.size()}")
            continue
        dd, df, cd, cf, fi, dp = run_both(qc, name)
        all_results.append((name, dd, df, cd, cf, fi, dp))
    except Exception as e:
        print(f"  FAILED: {e}")
        continue

print("\n" + "=" * 75)
print(f"{'Circuit':<22} {'DpD':>6} {'DpF':>6} "
      f"{'CostD':>9} {'CostF':>9} {'FidImpr':>8} {'DpPen':>7}")
print("-" * 75)
wins = 0
for r in all_results:
    name, dd, df, cd, cf, fi, dp = r
    print(f"{name:<22} {dd:>6.0f} {df:>6.0f} "
          f"{cd:>9.4f} {cf:>9.4f} {fi:>7.1f}% {dp:>6.1f}%")
    if fi > 0:
        wins += 1
print("-" * 75)
if all_results:
    avg_fi = np.mean([r[5] for r in all_results])
    avg_dp = np.mean([r[6] for r in all_results])
    print(f"{'Average':<22} {'':>6} {'':>6} "
          f"{'':>9} {'':>9} {avg_fi:>7.1f}% {avg_dp:>6.1f}%")
    print(f"\nFidelity wins: {wins}/{len(all_results)}")

FakeToronto 27-qubit heavy-hex
F_sqiswap: min=0.9338  max=0.9971  mean=0.9895  std=0.0125

Worst 5 qubits:
  logical 17 (physical 16): avg_F=0.9549
  logical 20 (physical 19): avg_F=0.9673
  logical 23 (physical 22): avg_F=0.9782
  logical 10 (physical 12): avg_F=0.9833
  logical  8 (physical 10): avg_F=0.9857
Best 5 qubits:
  logical  6 (physical  5): avg_F=0.9958
  logical  3 (physical  4): avg_F=0.9960
  logical  9 (physical  8): avg_F=0.9963
  logical 12 (physical 11): avg_F=0.9963
  logical  7 (physical  6): avg_F=0.9968


--- qft_n18 ---
  depth-sel:    depth=309  cost=8.3545  gates=1053
  fidelity-sel: depth=341  cost=8.5038  gates=1068
  → fid_impr=-1.8%  depth_pen=10.4%

--- qftentangled_n16 ---
  depth-sel:    depth=279  cost=8.3021  gates=852
  fidelity-sel: depth=284  cost=7.1490  gates=801
  → fid_impr=13.9%  depth_pen=1.8%

--- qpeexact_n16 ---
  depth-sel:    depth=315  cost=7.3152  gates=808
  fidelity-sel: depth=251  cost=5.9129  gates=763
  → fid_impr=19.2%  depth_pen